# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

## Using internal python implementation of WSPD class

In [1]:
from pathlib import Path

import tsplib95
import numpy as np

from wsp import ds, tsp

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.5
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [2]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [3]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using external cpython libs
still solving problem as they are seen

no bad pairs found up to 12k for s=1.5  (TSPLIB EUC2D)
no bad pairs found up to 12k for s=1.0 (TSPLIB EUC2D)

In [4]:
%load_ext line_profiler
from numba import njit
import elkai
from wspd import build_wspd_np, point as wsp_point # uses diam based sep factor
import wspd

SIZE_THRESHOLD2 = 500

In [5]:
all_problems : list[tsplib95.models.StandardProblem] = []

for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
    if file.suffix != ".tsp" or not file.is_file():
        continue
    problem = tsplib95.load(f"{file.absolute()}")
    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
        continue
    if problem.dimension > SIZE_THRESHOLD2 or problem.dimension <= 350: # Skip large TSPs
        continue
    all_problems.append(problem)

len(all_problems)

5

In [6]:
@njit(cache=True)
def wsp_heuristic_good(tour: np.ndarray, pos_in_tour: np.ndarray, A: np.ndarray, B: np.ndarray, state_arr: np.ndarray, S: float) -> bool:
    """A heuristic to check if a path is good based on the WSPs, checks if there are at least 2 connections between A and B"""
    assert len(A) > 0 and len(B) > 0, "Sets must be non-empty"

    exit_A, exit_B, biconn_count = 0, 0, 0
    num_points = len(pos_in_tour)

    state_arr[A] = 1
    state_arr[B] = 2

    for node in A:
        pos = pos_in_tour[node]

        v = state_arr[tour[pos + 1]] # since n+1 \in tour indices
        if v == 0: 
            exit_A += 1
        elif v == 2: 
            biconn_count += 1

        left_idx = num_points - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0: 
            exit_A += 1

    for node in B:
        pos = pos_in_tour[node]

        v = state_arr[tour[pos + 1]]
        if v == 0: 
            exit_B += 1
        elif v == 1: 
            biconn_count += 1

        left_idx = num_points - 1 if pos == 0 else pos - 1
        u = state_arr[tour[left_idx]]
        if u == 0: 
            exit_B += 1 

    state_arr[A] = 0
    state_arr[B] = 0

    # covers both single in outs and multi case
    if exit_A == 0 or exit_B == 0:
        return biconn_count == 2
    elif exit_A % 2 == 0:
        # this require s >= 1.5, all others require s >= 1.0
        if S >= 1.5:
            return biconn_count == 0
        return True
    else:
        return biconn_count == 1

In [7]:
def check_problem_tour(prob: tsplib95.models.StandardProblem, tour: np.ndarray, dim=2) -> tuple[int, int]:
    """Given a TSP and a wrapping tour, checks if the WSP heuristic holds"""
    points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]

    # Build the WSPDs with different separation factors
    #wspd_1: list[tuple[list[int], list[int]]] = build_wspd_np(len(points), dim, 1.0*2, points)
    print("built wspd1", flush=True)
    wspd_15: list[tuple[list[int], list[int]]] = build_wspd_np(len(points), dim, 1.5*2, points)
    print("built wspd1.5", flush=True)

    # pruning problems which cannot be bad
    wspd_1_conv = [] #filter(lambda x: len(x[0]) > 1 and len(x[1]) > 1, wspd_1)
    wspd_15_conv = filter(lambda x: len(x[0]) > 1 and len(x[1]) > 1, wspd_15)

    # vectorized node -> position map
    pos_in_tour = np.empty(len(tour) - 1, dtype=np.uint32)
    pos_in_tour[tour[:-1]] = np.arange(len(tour) - 1, dtype=np.uint32)

    state_arr = np.zeros(len(tour), dtype=np.uint8)

    biggest_pair = 0
    bad_count = 0
    for wspd, s in zip([wspd_1_conv, wspd_15_conv], [1.0, 1.5]):
        for A, B in wspd:
            biggest_pair = max(biggest_pair, min(len(A),len(B)))
            if not wsp_heuristic_good(tour, pos_in_tour, A, B, state_arr, s):
                bad_count += 1
                #coords = np.asarray([prob.node_coords[i] for i in prob.get_nodes()], dtype=np.float64)

                A_pts = np.asarray([prob.node_coords[i+1] for i in A], dtype=np.float32)
                B_pts = np.asarray([prob.node_coords[i+1] for i in B], dtype=np.float32)

                # exact min inter-cluster distance
                min_dist = np.linalg.norm(
                    A_pts[:, None, :] - B_pts[None, :, :], axis=2
                ).min()

                # exact cluster diameters
                diam_A = np.linalg.norm(
                    A_pts[:, None, :] - A_pts[None, :, :], axis=2
                ).max()
                diam_B = np.linalg.norm(
                    B_pts[:, None, :] - B_pts[None, :, :], axis=2
                ).max()

                true_sep = min_dist / max(diam_A, diam_B)

                print(
                    f"{prob.name}: bad pair ({A},{B}), true separation={true_sep:.2f} "
                    #f"(min_dist={min_dist:.2f}, diamA={diam_A:.2f}, diamB={diam_B:.2f})"
                )
    return bad_count, biggest_pair

In [8]:
#def profile_bottleneck(all_problems: list[tsplib95.models.StandardProblem]):

for prob in all_problems:
    print(f"Checking {prob.name}...", flush=True)

    lkh_instance = elkai.Coordinates2D({
        i-1: prob.node_coords[i] for i in prob.get_nodes()
    })

    # tour numpy conversions
    tour_lkh = lkh_instance.solve_tsp(runs=1)
    tour = np.array(tour_lkh, dtype=np.uint16)

    check_problem_tour(prob, tour)

#%lprun -f profile_bottleneck -u 1.0 profile_bottleneck(all_problems)
print("Done")

Checking d493...
built wspd1
built wspd1.5
Checking fl417...
built wspd1
built wspd1.5
Checking pcb442...
built wspd1
built wspd1.5
Checking pr439...
built wspd1
built wspd1.5
Checking rd400...
built wspd1
built wspd1.5
Done


## Loading previously solved TSPs

In [9]:
gaia_prob = tsplib95.load("ALL_tsp/gaia2079471.tsp")
gaia_tour_prob = tsplib95.load("ALL_tsp/gaia2079471.tour")

In [10]:
gaia_tour = np.array(gaia_tour_prob.tours[0] + [1], dtype=np.uint32) - 1
gaia_tour.shape

(2079472,)

In [12]:
check_problem_tour(gaia_prob, gaia_tour, dim=3)

#%lprun -f check_problem_tour -u 1.0 check_problem_tour(gaia_prob, gaia_tour, dim=3)

built wspd1
built wspd1.5


(0, 13507)